第二版，导入MCC ESD的表。

如果是query chip出现NA，~~会动态的调整fit模型时用到的参数~~（加了自定义权重之后这个功能暂时没办法实现）

如果是candidate chip出现NA，会用中位数代替。

In [96]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

df=pd.read_csv(
    'products/MCC/raw/MCC_DataExport_esd-protection-devices(MCC-esd-protection-devices).csv',
    header=0,
    skiprows=[0,2]
)

df.columns = [' '.join(col.split()) for col in df.columns]

# 单独去掉ESD0512LB，他有两种参数，如果一起放进df里会直接把float coerce成string
df = df[df['Product'] != 'ESD0512LB']

df_selected=df.iloc[:,[1,3]+list(range(4,17))]
df_selected.info()

<class 'pandas.DataFrame'>
Index: 543 entries, 0 to 543
Data columns (total 15 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Product                                543 non-null    str    
 1   Compliance                             543 non-null    str    
 2   Number of Functions                    543 non-null    int64  
 3   Configuration                          543 non-null    str    
 4   Package Type                           543 non-null    str    
 5   Reverse Standoff Voltage VRWM(V)       543 non-null    str    
 6   Peak Pulse Current IPP(A)              543 non-null    float64
 7   Max. Clamping Voltage VC (V)           543 non-null    float64
 8   Junction Capacitance CJ(pF)            539 non-null    float64
 9   Peak Pluse Power Dissipation PPPK (W)  542 non-null    float64
 10  Maximum Reverse Leakage IR (uA)        543 non-null    float64
 11  Breakdown Voltage Min 

In [97]:
def apply_hard_constraints(candidate_chips, query_chip): 
    # 从 query_chip 中提取标量值
    num_functions = query_chip['Number of Functions'].iloc[0]
    config = query_chip['Configuration'].iloc[0]
    package = query_chip['Package Type'].iloc[0]

    # """
    # 硬约束过滤：
    # 1. 配置、封装类型 必须完全匹配
    # 2. VRWM(替料) <= VRWM(原料)
    # 3. VC(替料) <= VC(原料)
    # 4. IPP(替料) >= IPP(原料)
    # 5. CJ(替料) <= CJ(原料)
    # """   
    # # 提取电气参数（使用 .iloc[0] 确保是标量）
    # vrwm_query = query_chip['Reverse Standoff Voltage VRWM(V)'].iloc[0]
    # vc_query = query_chip['Max. Clamping Voltage VC (V)'].iloc[0]
    # ipp_query = query_chip['Peak Pulse Current IPP(A)'].iloc[0]
    # cj_query = query_chip['Junction Capacitance CJ(pF)'].iloc[0]

    filtered = candidate_chips[
        (candidate_chips['Number of Functions'] == num_functions) &
        (candidate_chips['Configuration'] == config) &
        (candidate_chips['Package Type'] == package)
    ].copy()

    # filtered = candidate_chips[
    #     (candidate_chips['Number of Functions'] == num_functions) &
    #     (candidate_chips['Configuration'] == config) &
    #     (candidate_chips['Package Type'] == package) &
    #     (candidate_chips['Reverse Standoff Voltage VRWM(V)'] <= vrwm_query) &
    #     (candidate_chips['Max. Clamping Voltage VC (V)'] <= vc_query) &
    #     (candidate_chips['Peak Pulse Current IPP(A)'] >= ipp_query) &
    #     (candidate_chips['Junction Capacitance CJ(pF)'] <= cj_query)
    # ].copy()
    
    
    return filtered

In [98]:
all_numeric_features = [
    'Reverse Standoff Voltage VRWM(V)',
    'Peak Pulse Current IPP(A)',
    'Max. Clamping Voltage VC (V)',
    'Junction Capacitance CJ(pF)',
    'Peak Pluse Power Dissipation PPPK (W)',
    'Maximum Reverse Leakage IR (uA)',
    'Breakdown Voltage Min VBR(V)',
    'Breakdown Voltage Max VBR(V)',
    'Junction Temperature Tj [max] (°C)'
]

categorical_features = ['VESDIEC61000-4-2 Air/Contact (kV)']

# 非特征的列（不参与相似度计算）
exclude_columns = ['Product', 'Number of Functions', 'Configuration', 'Package Type']

In [99]:
def get_available_features(query_chip, all_features):
    """
    检查 query_chip 中哪些特征列不是 NA，返回可用特征列表
    """
    available = []
    for col in all_features:
        if col in query_chip.columns:
            # 检查该列的第一个值是否为 NA
            if pd.notna(query_chip[col].iloc[0]):
                available.append(col)
            else:
                print(f"⚠️ 跳过特征 '{col}'（query_chip 中为 NA）")
        else:
            print(f"⚠️ 跳过特征 '{col}'（不在 query_chip 中）")
    return available

In [100]:
def create_dynamic_preprocessor(df_inventory, query_chip, all_features):
    """
    根据 query_chip 的缺失情况，动态创建预处理流水线
    """
    # 获取可用的特征
    available_features = get_available_features(query_chip, all_features)
    
    if not available_features:
        raise ValueError("query_chip 中没有任何可用特征！")
    
    print(f"✅ 使用的数值特征 ({len(available_features)}个):", available_features)
    
    # 创建预处理器（只对可用特征进行预处理）
    preprocessor = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  # 候选芯片缺失值用中位数填充
            ('scaler', StandardScaler())
        ]), available_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])
    
    # 在完整数据集上 fit
    preprocessor.fit(df_inventory)
    
    return preprocessor, available_features

In [101]:
def create_query_chip(columns, values_dict):
    """
    创建 query_chip，缺失值用 np.nan 表示
    """
    # 从 df_selected 获取列名
    columns_list = columns.tolist() if hasattr(columns, 'tolist') else columns
    
    # 按照列顺序构建值列表
    values = []
    for col in columns_list:
        if col in values_dict:
            values.append(values_dict[col])
        else:
            values.append(np.nan)  # 没有指定的列用 NA
    
    return pd.DataFrame([values], columns=columns_list)

In [102]:
# def weighted_similarity(query_vec, candidate_matrix, feature_weights):
#     """
#     计算加权余弦相似度
#     feature_weights: 列表，顺序与编码后的特征列顺序一致
#     """
#     # 先对候选矩阵的每一列乘以对应的权重
#     weighted_candidates = candidate_matrix * feature_weights
#     weighted_query = query_vec * feature_weights
    
#     # 计算余弦相似度
#     similarities = cosine_similarity(weighted_query, weighted_candidates)
#     return similarities.flatten()

from scipy import sparse

def weighted_similarity(query_vec, candidate_matrix, feature_weights):
    feature_weights = np.asarray(feature_weights)

    if sparse.issparse(query_vec):
        weighted_query = query_vec.multiply(feature_weights)
    else:
        weighted_query = query_vec * feature_weights

    if sparse.issparse(candidate_matrix):
        weighted_candidates = candidate_matrix.multiply(feature_weights)
    else:
        weighted_candidates = candidate_matrix * feature_weights

    similarities = cosine_similarity(
        weighted_query,
        weighted_candidates
    )

    return similarities.flatten()

In [103]:
def recommend_alternatives_dynamic(df_inventory, query_chip, all_features, top_k=10, weights=None):
    """
    动态推荐函数 - 根据 query_chip 的可用特征自动调整
    """
    # 1. 硬约束过滤
    candidates = apply_hard_constraints(df_inventory, query_chip)
    if candidates.empty:
        return "没有配置/封装完全匹配的替代品"
    
    # 2. 动态创建预处理器
    preprocessor, available_features = create_dynamic_preprocessor(
        df_inventory, query_chip, all_features
    )
    
    # 3. 编码
    query_vec = preprocessor.transform(query_chip)
    # print(f"编码后的总特征数: {query_vec.shape[1]}")
    candidate_vecs = preprocessor.transform(candidates)
    
    # 4. 计算相似度
    if weights is None:
        weights = np.ones(candidate_vecs.shape[1])
    else:
        # 如果传入的权重长度与特征数不匹配，调整
        if len(weights) != candidate_vecs.shape[1]:
            print(f"⚠️ 权重长度 ({len(weights)}) 与特征数 ({candidate_vecs.shape[1]}) 不匹配，使用默认权重")
            weights = np.ones(candidate_vecs.shape[1])
    
    scores = weighted_similarity(query_vec, candidate_vecs, weights)
    
    # 5. 排序并返回
    candidates = candidates.copy()
    candidates['similarity_score'] = scores
    results = candidates.sort_values('similarity_score', ascending=False).head(top_k)

    # 添加车规级标识列
    # 假设 df_inventory 中有 'Compliance' 列
    results['Is_Automotive'] = results['Compliance'].apply(
        lambda x: '车规级' if pd.notna(x) and 'A' in str(x) else '非车规级'
    )
    
    # 返回结果（包含动态特征）
    return_columns = ['Product', 'similarity_score', "Is_Automotive"] + available_features + categorical_features
    return results[return_columns]

# def recommend_alternatives_dynamic(df_inventory, query_chip, all_features, top_k=10, weights=None):
#     """
#     动态推荐函数 - 根据 query_chip 的可用特征自动调整
#     """
#     # 1. 硬约束过滤
#     candidates = apply_hard_constraints(df_inventory, query_chip)
#     if candidates.empty:
#         return "没有配置/封装完全匹配的替代品"
    
#     # 2. 动态创建预处理器
#     preprocessor, available_features = create_dynamic_preprocessor(
#         df_inventory, query_chip, all_features
#     )
    
#     # 3. 编码
#     query_vec = preprocessor.transform(query_chip)
#     candidate_vecs = preprocessor.transform(candidates)
    
#     # 4. 自动生成权重
#     if weights is None:
#         weights = np.ones(candidate_vecs.shape[1])
#     else:
#         # 如果传入的权重长度与特征数不匹配，自动补全
#         if len(weights) != candidate_vecs.shape[1]:
#             n_numeric = len(available_features)  # 数值特征数量（如 9）
#             n_categorical = candidate_vecs.shape[1] - n_numeric  # 分类特征数量
            
#             print(f"📊 总特征数: {candidate_vecs.shape[1]}, 数值特征: {n_numeric}, 分类特征: {n_categorical}")
            
#             if len(weights) == n_numeric:
#                 # 只传了数值特征的权重，自动补充分类特征权重
#                 categorical_weight = 0.3
#                 weights = np.concatenate([
#                     np.array(weights),
#                     np.full(n_categorical, categorical_weight)
#                 ])
#                 print(f"✅ 自动补全权重: 数值权重 {len(weights) - n_categorical} 个, 分类权重 {n_categorical} 个 (全部设为 {categorical_weight})")
#             else:
#                 # 长度完全不匹配，使用默认权重
#                 print(f"⚠️ 权重长度 ({len(weights)}) 与总特征数 ({candidate_vecs.shape[1]}) 不匹配，使用默认权重")
#                 weights = np.ones(candidate_vecs.shape[1])
    
#     scores = weighted_similarity(query_vec, candidate_vecs, weights)
    
#     # 5. 排序并返回
#     candidates = candidates.copy()
#     candidates['similarity_score'] = scores
#     results = candidates.sort_values('similarity_score', ascending=False).head(top_k)

#     # 添加车规级标识列
#     results['Is_Automotive'] = results['Compliance'].apply(
#         lambda x: '车规级' if pd.notna(x) and 'A' in str(x) else '非车规级'
#     )
    
#     # 返回结果（包含动态特征）
#     return_columns = ['Product', 'similarity_score', 'Is_Automotive'] + available_features + categorical_features
#     return results[return_columns]

In [ ]:
# 构建 query_chip（所有参数都有值）
columns = df_selected.columns.tolist()

# query_values = { #ESD24VT2BHE3
#     'Product': 'NUP2105LT1G',
#     'Number of Functions': 2,
#     'Configuration': 'Bidirectional',
#     'Package Type': 'SOT-23',
#     'Reverse Standoff Voltage VRWM(V)': 24,
#     'Peak Pulse Current IPP(A)': 8.0,
#     'Max. Clamping Voltage VC (V)': 44,
#     'Junction Capacitance CJ(pF)': 30,
#     'Peak Pluse Power Dissipation PPPK (W)': 350,
#     'Maximum Reverse Leakage IR (uA)': 0.1,
#     'Breakdown Voltage Min VBR(V)': 26.2,
#     'Breakdown Voltage Max VBR(V)': 32,
#     'Junction Temperature Tj [max] (°C)': 150,
#     'VESDIEC61000-4-2 Air/Contact (kV)': '±30'
# }

# query_values = { # ESDLC24VT2BHE3
#     'Product': 'ESD2CANFD24T2Q',
#     'Number of Functions': 2,
#     'Configuration': 'Bidirectional',
#     'Package Type': 'SOT-23',
#     'Reverse Standoff Voltage VRWM(V)': 24,
#     'Peak Pulse Current IPP(A)': 3.5,
#     'Max. Clamping Voltage VC (V)': 36,
#     'Junction Capacitance CJ(pF)': 2.5,
#     'Peak Pluse Power Dissipation PPPK (W)': 133,
#     'Maximum Reverse Leakage IR (uA)': 0.05,
#     'Breakdown Voltage Min VBR(V)': 25.5,
#     'Breakdown Voltage Max VBR(V)': 35.5,
#     'Junction Temperature Tj [max] (°C)': 150,
#     'VESDIEC61000-4-2 Air/Contact (kV)': '±25'
# }

# query_values = { # ESDSB5V0T2BHE3
#     'Product': 'ESD0502EBQ',
#     'Number of Functions': 2,
#     'Configuration': 'Bidirectional',
#     'Package Type': 'SOT-23',
#     'Reverse Standoff Voltage VRWM(V)': 5,
#     'Peak Pulse Current IPP(A)': 24,
#     'Max. Clamping Voltage VC (V)': 13,
#     'Junction Capacitance CJ(pF)': 50,
#     'Peak Pluse Power Dissipation PPPK (W)': 312,
#     'Maximum Reverse Leakage IR (uA)': 1,
#     'Breakdown Voltage Min VBR(V)': 6,
#     'Breakdown Voltage Max VBR(V)': 7.8,
#     'Junction Temperature Tj [max] (°C)': 150,
#     'VESDIEC61000-4-2 Air/Contact (kV)': '±30'
# }

# query_values = { # 
#     'Product': 'TPD2E2U06-Q1',
#     'Number of Functions': 2,
#     'Configuration': 'Unidirectional',
#     'Package Type': 'SOT-23',
#     'Reverse Standoff Voltage VRWM(V)': 5.5,
#     'Peak Pulse Current IPP(A)': 5,
#     'Max. Clamping Voltage VC (V)': 12.4,
#     'Junction Capacitance CJ(pF)': None,
#     'Peak Pluse Power Dissipation PPPK (W)': None,
#     'Maximum Reverse Leakage IR (uA)': 0.01,
#     'Breakdown Voltage Min VBR(V)': 6.5,
#     'Breakdown Voltage Max VBR(V)': 8.5,
#     'Junction Temperature Tj [max] (°C)': 125,
#     'VESDIEC61000-4-2 Air/Contact (kV)': '±30'
# }

query_chip = create_query_chip(columns, query_values)

# 权重
# 'Reverse Standoff Voltage VRWM(V)': 2.0,   # 硬约束参数，非常重要
# 'Peak Pulse Current IPP(A)': 1.8,          # 硬约束参数，重要
# 'Max. Clamping Voltage VC (V)': 2.0,       # 硬约束参数，非常重要
# 'Junction Capacitance CJ(pF)': 1.8,        # 硬约束参数，重要
# 'Peak Pluse Power Dissipation PPPK (W)': 1.0,  # 参考值
# 'Maximum Reverse Leakage IR (uA)': 0.8,    # 不那么关键
# 'Breakdown Voltage Min VBR(V)': 1.2,       # 与VRWM相关，次重要
# 'Breakdown Voltage Max VBR(V)': 1.2,       # 与VRWM相关，次重要
# 'Junction Temperature Tj [max] (°C)': 0.5  # 大多数都是150，区分度低
custom_weights = np.array([2.0, 1.8, 2.0, 1.5, 1.0, 0.8, 1.2, 1.2, 0.5])

# 执行推荐（所有9个数值特征都会使用）
result = recommend_alternatives_dynamic(df_selected, query_chip, all_numeric_features, top_k=10)
print(result)

⚠️ 跳过特征 'Junction Capacitance CJ(pF)'（query_chip 中为 NA）
⚠️ 跳过特征 'Peak Pluse Power Dissipation PPPK (W)'（query_chip 中为 NA）
✅ 使用的数值特征 (7个): ['Reverse Standoff Voltage VRWM(V)', 'Peak Pulse Current IPP(A)', 'Max. Clamping Voltage VC (V)', 'Maximum Reverse Leakage IR (uA)', 'Breakdown Voltage Min VBR(V)', 'Breakdown Voltage Max VBR(V)', 'Junction Temperature Tj [max] (°C)']
         Product  similarity_score Is_Automotive  \
200        HSM05          0.937239          非车规级   
106       SM3.3H          0.851786          非车规级   
208  ESDULC5V0T2          0.702803          非车规级   
209    ESDH12VT2          0.588634          非车规级   
410     MMBZ6V2C          0.532492          非车规级   
229     MMBZ6V2A          0.532492          非车规级   
230     MMBZ6V8A          0.511512          非车规级   
411     MMBZ6V8C          0.511512          非车规级   
387    ESD3V3T2Q          0.477518           车规级   
395   ESD3V3ET2Q          0.477518           车规级   

    Reverse Standoff Voltage VRWM(V)  Peak Pulse Curre